# Construction du dataset des écarts de prix — Compagnon Immobilier

**Périmètre : 2016–2025 | Sortie : `ecarts_prix_equipements_2016_2025.csv`**

Ce notebook rassemble dans un seul document :
- l'audit des sources et des jointures amont disponibles,
- la préparation des prix communaux,
- le calcul du dataset ecarts_prix_equipements_2016_2025

## 1. Sources et chaîne de données

- **Table cible immobilière** `cible_commune_annee_type.csv` : une ligne par commune et année, avec les effectifs de ventes et les prix médians au m² des maisons et appartements.
- **Table consolidée BPE** `cible_commune_annee_type_BPE_152_indicateurs.csv.gz` : reprend la cible immobilière et ajoute le millésime BPE, les indicateurs de présence/absence/inconnu et les informations de contrôle géographique. C'est la table consolidée utilisée pour les six équipements de ce travail.
- **Dataset de sortie** `ecarts_prix_equipements_2016_2025.csv` : une ligne par année × type de bien × équipement (120 lignes attendues).

### Jointures amont : ce qui est établi et ce qui ne l'est pas

La consolidation disponible associe la cible immobilière à des indicateurs BPE sur la base de la commune et du millésime BPE applicable (BPE 2015 pour les années 2016–2020 ; BPE 2020 pour 2021–2025). Les contrôles géographiques et le statut de correspondance sont documentés dans les colonnes `bpe_statut_jointure_initial`, `code_commune_2026` et `bpe_code_mapping_status` lorsqu'elles sont présentes.


## 2. Charger les sources et contrôler leurs clés

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

ANNEES = list(range(2016, 2026))
SEUIL_VENTES = 3
Q_BAS, Q_HAUT = 0.01, 0.99
SEUIL_COMMUNES_CARTE = 50

# Recherche automatique des fichiers, y compris après téléversement dans Colab.
def trouver_fichier(nom, extensions=None):
    candidats = list(Path.cwd().rglob(nom))
    if candidats:
        return candidats[0]
    try:
        from google.colab import files
        print(f"Téléverse le fichier : {nom}")
        uploaded = files.upload()
        candidats = [Path(n) for n in uploaded if Path(n).name == nom or (extensions and n.endswith(tuple(extensions)))]
        if candidats:
            return candidats[0]
    except ImportError:
        pass
    raise FileNotFoundError(f"Fichier introuvable : {nom}. Place-le à côté du notebook ou téléverse-le.")

NOM_CIBLE = "cible_commune_annee_type.csv"
NOM_CONSOLIDEE = "cible_commune_annee_type_BPE_152_indicateurs.csv.gz"
CHEMIN_CIBLE = trouver_fichier(NOM_CIBLE)
CHEMIN_CONSOLIDEE = trouver_fichier(NOM_CONSOLIDEE, extensions=[".csv.gz", ".gz"])

cible = pd.read_csv(CHEMIN_CIBLE, dtype={"code_commune": "string"}, low_memory=False)
consolidee = pd.read_csv(CHEMIN_CONSOLIDEE, dtype={"code_commune": "string", "code_commune_2026": "string"}, low_memory=False)
for df in (cible, consolidee):
    df["code_commune"] = df["code_commune"].astype("string").str.strip()
    df["annee"] = pd.to_numeric(df["annee"], errors="coerce").astype("Int64")

print("Cible immobilière :", cible.shape)
print("Table consolidée BPE :", consolidee.shape)
print("Période cible :", cible.annee.min(), "–", cible.annee.max())
display(cible.head())
display(consolidee[[c for c in ["code_commune", "annee", "bpe_annee_source", "bpe_statut_jointure_initial", "code_commune_2026", "bpe_code_mapping_status"] if c in consolidee.columns]].head())


Téléverse le fichier : cible_commune_annee_type.csv


## 3. Auditer puis matérialiser la jointure cible ↔ table BPE consolidée

La table consolidée contient déjà les colonnes de prix et d'effectifs. Pour rendre la relation explicite et vérifiable, on compare les clés puis on joint les colonnes BPE à la cible immobilière. Les indicateurs sont sélectionnés depuis la table consolidée ; la jointure des fichiers bruts DVF/BPE n'est pas réinventée ici.


In [ ]:
# Colonnes immobilières attendues dans la table cible.
colonnes_cible = ["code_commune", "annee", "n_maison", "n_appart", "prix_m2_med_maison", "prix_m2_med_appart"]
absentes = [c for c in colonnes_cible if c not in cible.columns]
if absentes:
    raise ValueError(f"Colonnes manquantes dans la cible : {absentes}")

# Vérifier l'unicité des clés avant jointure.
print("Doublons cible (commune, année) :", int(cible.duplicated(["code_commune", "annee"]).sum()))
print("Doublons consolidée (commune, année) :", int(consolidee.duplicated(["code_commune", "annee"]).sum()))

colonnes_bpe = [c for c in consolidee.columns if c.startswith("bpe_presence_")]
colonnes_meta = [c for c in ["bpe_annee_source", "bpe_statut_jointure_initial", "bpe_age_vintage_ans_initial", "code_commune_2026", "bpe_code_mapping_status", "bpe_nb_types_presents", "bpe_nb_types_absents", "bpe_nb_types_inconnus"] if c in consolidee.columns]
# N'inclure que les colonnes BPE/métadonnées dans le côté droit de la jointure.
bpe_a_joindre = consolidee[["code_commune", "annee"] + colonnes_meta + colonnes_bpe].copy()

# La validation one_to_one évite une multiplication accidentelle des lignes.
source = cible.merge(bpe_a_joindre, on=["code_commune", "annee"], how="left", validate="one_to_one", indicator="statut_jointure_cible_bpe")
print("Dimensions après jointure :", source.shape)
display(source["statut_jointure_cible_bpe"].value_counts(dropna=False).rename_axis("statut").to_frame("lignes"))
print("Taux de correspondance :", round(source["statut_jointure_cible_bpe"].eq("both").mean()*100, 2), "%")
print("Nombre d'indicateurs BPE disponibles :", len(colonnes_bpe))

# Contrôle de cohérence : les prix/effectifs joints doivent correspondre à la cible d'origine.
for col in ["n_maison", "n_appart", "prix_m2_med_maison", "prix_m2_med_appart"]:
    if col in consolidee.columns:
        comparaison = source[col].reset_index(drop=True).equals(consolidee[col].reindex(source.index).reset_index(drop=True))
        print(f"Contrôle {col} : comparer avec prudence (l'ordre des sources peut différer) ; identique par index = {comparaison}")

# On conserve la source consolidée comme table de travail canonique, car elle est déjà documentée
# et contient les colonnes exactes de la cible + BPE utilisées dans le projet.
source = consolidee.copy()


## 3. Définir les équipements et les variables utilisées

| Code BPE | Équipement | Variable source |
|---|---|---|
| `a206` | Bureau de poste | `bpe_presence_a206` |
| `b207` | Boulangerie-pâtisserie | `bpe_presence_b207` |
| `d502` | Accueil du jeune enfant (EAJE) | `bpe_presence_d502` |
| `d101` | Établissement de santé — court séjour | `bpe_presence_d101` |
| `d107` | Maternité | `bpe_presence_d107` |
| `d113` | Maison de santé pluriprofessionnelle | `bpe_presence_d113` |

Les variables immobilières utilisées sont l'année, les nombres de ventes et les prix médians au m² pour les maisons et les appartements. Les indicateurs BPE prennent trois états : `1` présent, `0` absent, `NaN` inconnu/non renseigné. **Les valeurs inconnues ne sont jamais assimilées à une absence.**

## 4. Préparer les prix immobiliers

In [ ]:
# Une ligne par commune et année. Les deux types de biens sont traités séparément.
maisons = source[["code_commune", "annee", "n_maison", "prix_m2_med_maison", "bpe_annee_source"] +
                 [f"bpe_presence_{c}" for c in ["a206", "b207", "d502", "d101", "d107", "d113"]]].copy()
maisons = maisons.rename(columns={"n_maison": "n_ventes", "prix_m2_med_maison": "prix_m2_med"})
maisons["type_bien"] = "Maisons"

appartements = source[["code_commune", "annee", "n_appart", "prix_m2_med_appart", "bpe_annee_source"] +
                      [f"bpe_presence_{c}" for c in ["a206", "b207", "d502", "d101", "d107", "d113"]]].copy()
appartements = appartements.rename(columns={"n_appart": "n_ventes", "prix_m2_med_appart": "prix_m2_med"})
appartements["type_bien"] = "Appartements"

long = pd.concat([maisons, appartements], ignore_index=True)
long = long[long["annee"].isin(ANNEES)].copy()
long = long[long["n_ventes"] >= SEUIL_VENTES].copy()
long = long[long["prix_m2_med"].notna() & (long["prix_m2_med"] > 0)].copy()

# Exclusion des valeurs extrêmes : les observations hors des quantiles 1 % et 99 %
# sont retirées dans chaque couple année/type de bien (elles ne sont pas winsorisées).
def filtrer_extremes(g):
    bas = g["prix_m2_med"].quantile(Q_BAS)
    haut = g["prix_m2_med"].quantile(Q_HAUT)
    return g[g["prix_m2_med"].between(bas, haut, inclusive="both")]

long = (long.groupby(["annee", "type_bien"], group_keys=False)
            .apply(filtrer_extremes, include_groups=True)
            .reset_index(drop=True))

print(f"Observations conservées après filtrage : {len(long):,}")
display(long.groupby(["type_bien", "annee"]).size().unstack(0))

### Interprétation du filtrage

- La période est limitée à 2016–2025.
- Les communes doivent compter au moins trois ventes pour le type de bien considéré.
- Les prix manquants ou non positifs sont exclus.
- Les observations situées hors des quantiles 1 % et 99 % sont retirées séparément pour chaque année et type de bien. Il s'agit d'une exclusion des valeurs extrêmes, et non d'un remplacement des valeurs.

Ces règles réduisent l'influence des observations atypiques, mais elles modifient aussi la population de communes analysée. Les effectifs sont donc conservés dans le dataset de sortie.

## 5. Calculer les écarts par équipement, année et type de bien

In [ ]:
EQUIPEMENTS = {
    "a206": "Bureau de poste",
    "b207": "Boulangerie-pâtisserie",
    "d502": "Accueil du jeune enfant (EAJE)",
    "d101": "Établissement de santé — court séjour",
    "d107": "Maternité",
    "d113": "Maison de santé pluriprofessionnelle",
}

resultats = []
for (annee, type_bien), groupe_annee in long.groupby(["annee", "type_bien"], sort=True):
    for code, libelle in EQUIPEMENTS.items():
        col = f"bpe_presence_{code}"
        # Ne garder que les états explicitement connus (0 ou 1)
        connu = groupe_annee[groupe_annee[col].isin([0, 1])].copy()
        present = connu[connu[col].eq(1)]["prix_m2_med"]
        absent = connu[connu[col].eq(0)]["prix_m2_med"]
        n_present, n_absent = int(len(present)), int(len(absent))
        med_present = float(present.median()) if n_present else np.nan
        med_absent = float(absent.median()) if n_absent else np.nan
        ecart = med_present - med_absent if n_present and n_absent else np.nan
        pct = (ecart / med_absent * 100) if pd.notna(ecart) and med_absent != 0 else np.nan
        n_total = len(groupe_annee)
        couverture = ((n_present + n_absent) / n_total * 100) if n_total else np.nan
        millesime = groupe_annee["bpe_annee_source"].dropna()
        bpe_source = int(millesime.mode().iloc[0]) if not millesime.empty else np.nan
        resultats.append({
            "annee": int(annee), "type_bien": type_bien, "code_equipement": code,
            "equipement": libelle, "n_present": n_present, "n_absent": n_absent,
            "mediane_present_eur_m2": med_present, "mediane_absent_eur_m2": med_absent,
            "ecart_eur_m2": ecart, "ecart_pct_vs_absent": pct,
            "couverture_connue_pct": couverture, "bpe_annee_source": bpe_source
        })

ecarts = pd.DataFrame(resultats).sort_values(["annee", "type_bien", "code_equipement"]).reset_index(drop=True)
print("Dimensions du dataset final :", ecarts.shape)
display(ecarts.head(12))

### Définition de l'écart

`ecart_eur_m2 = médiane(prix au m² | équipement présent) − médiane(prix au m² | équipement absent)`.

- Écart positif : prix médian supérieur dans le groupe équipé.
- Écart négatif : prix médian inférieur dans le groupe équipé.
- Écart manquant : un des deux groupes ne permet pas de calculer la médiane.

L'écart est descriptif et non causal. Les communes ne sont pas ajustées pour leurs autres caractéristiques socio-économiques ou géographiques.

## 6. Exporter le dataset d'écarts

In [ ]:
NOM_SORTIE = "ecarts_prix_equipements_2016_2025.csv"
ecarts.to_csv(NOM_SORTIE, index=False, encoding="utf-8-sig")
print("Dataset enregistré :", Path(NOM_SORTIE).resolve())
print(f"{len(ecarts)} lignes ; {ecarts['annee'].nunique()} années ; "
      f"{ecarts['type_bien'].nunique()} types de biens ; {ecarts['equipement'].nunique()} équipements")

## 8. Contrôles finaux

In [ ]:
# Vérifications de structure et d'intégrité
assert len(ecarts) == 120, f"120 lignes attendues, {len(ecarts)} obtenues"
assert ecarts["annee"].nunique() == 10
assert ecarts["type_bien"].nunique() == 2
assert ecarts["equipement"].nunique() == 6
assert set(ecarts["annee"].unique()) == set(ANNEES)

print("Contrôles OK : 120 lignes (10 années × 2 types de biens × 6 équipements).")
print("Millésimes BPE observés :")
display(ecarts.groupby(["annee", "bpe_annee_source"]).size().unstack(fill_value=0))
print("Fichiers produits :")
for f in [NOM_SORTIE, "carte_thermique_maisons.png", "carte_thermique_appartements.png"]:
    print("-", Path(f).resolve())